# Computational cost — Appendix C


In [ ]:
import pandas as pd, json
from pathlib import Path

RAG_RUNS = Path("rag_runs_random")
REVIEW_OUTPUTS = Path("review_outputs_v6")

def load(root):
    for p in sorted(root.rglob("*.json")):
        yield json.load(p.open())

def tok(d, k):
    return (d.get("token_usage") or {}).get(k, 0)

def sec(d):
    return (d.get("timing") or {}).get("total_duration_seconds", 0.0)

gen = pd.DataFrame([{"prompt": tok(d, "prompt_tokens"), "total": tok(d, "total_tokens"),
                     "seconds": sec(d)}
                    for d in load(RAG_RUNS) if "token_usage" in d and "generation_model" in d])

rev = pd.DataFrame([{"paper": d.get("paper_id", "?"), "prompt": tok(c, "prompt_tokens"),
                     "total": tok(c, "total_tokens"), "seconds": sec(c)}
                    for d in load(REVIEW_OUTPUTS) if "reviews" in d
                    for c in [x.get("official_review") or {} for x in d["reviews"]]
                             + ([d["meta_review"]] if d.get("meta_review") else [])])

assert not gen.empty and not rev.empty, "no records matched - check the paths above"
PAPERS = rev.paper.nunique()
print(f"{len(gen):,} section calls | {len(rev):,} review calls | {PAPERS:,} papers")

8,400 section calls | 5,600 review calls | 1,400 papers


## Table C.1 — Aggregate cost

In [2]:
cost_aggregate = pd.DataFrame([
    {"Pipeline": "Generation", "Calls": len(gen), "Total tokens": gen.total.sum(),
     "Prompt share": gen.prompt.sum() / gen.total.sum(),
     "Model time (h)": gen.seconds.sum() / 3600},
    {"Pipeline": "Review", "Calls": len(rev), "Total tokens": rev.total.sum(),
     "Prompt share": rev.prompt.sum() / rev.total.sum(),
     "Model time (h)": rev.seconds.sum() / 3600},
])
cost_aggregate.loc[2] = {
    "Pipeline": "Total", "Calls": cost_aggregate.Calls.sum(),
    "Total tokens": cost_aggregate["Total tokens"].sum(),
    "Prompt share": (gen.prompt.sum() + rev.prompt.sum()) / cost_aggregate["Total tokens"].sum(),
    "Model time (h)": cost_aggregate["Model time (h)"].sum(),
}
cost_aggregate = cost_aggregate.round({"Prompt share": 2, "Model time (h)": 1})

cost_aggregate

,Pipeline,Calls,Total tokens,Prompt share,Model time (h)
0,Generation,8400,38583654,0.56,73.7
1,Review,5600,50952628,0.94,36.3
2,Total,14000,89536282,0.78,109.9


## Table C.2 — Cost per paper

In [3]:
cost_per_paper = pd.DataFrame([
    {"Stage": "Generation (6 sections)",
     "Tokens per paper": gen.total.sum() / PAPERS,
     "Seconds per paper": gen.seconds.sum() / PAPERS},
    {"Stage": "Review (3 reviewers + Area Chair)",
     "Tokens per paper": rev.total.sum() / PAPERS,
     "Seconds per paper": rev.seconds.sum() / PAPERS},
])
cost_per_paper.loc[2] = {"Stage": "Full cycle",
                         "Tokens per paper": cost_per_paper["Tokens per paper"].sum(),
                         "Seconds per paper": cost_per_paper["Seconds per paper"].sum()}
cost_per_paper = cost_per_paper.round(1)

cost_per_paper

,Stage,Tokens per paper,Seconds per paper
0,Generation (6 sections),27559.8,189.4
1,Review (3 reviewers + Area Chair),36394.7,93.2
2,Full cycle,63954.5,282.6
